# AccessGuard — Notebook 05: ONNX Export & Artifact Packaging

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

Exports trained PyTorch weights to ONNX format with dynamic batching,
optimizes using onnxslim, and packages artifacts for deployment into local AccessGuard.


In [ ]:
!pip install -q ultralytics onnx onnxslim onnxruntime


In [ ]:
# Export YOLOv8 model to ONNX
from ultralytics import YOLO
import os

model = YOLO('yolov8n.pt')
export_path = model.export(format='onnx', opset=18, dynamic=True)
print(f'Exported ONNX model to: {export_path}')


In [ ]:
# Verify ONNX Runtime inference
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession(export_path, providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
dummy_tensor = np.zeros((1, 3, 640, 640), dtype=np.float32)
outputs = session.run(None, {input_name: dummy_tensor})
print(f'Inference output shape: {outputs[0].shape}')


In [ ]:
# Package model artifact for local AccessGuard
import tarfile

artifact_name = 'accessguard_yolov8n_onnx.tar.gz'
with tarfile.open(artifact_name, 'w:gz') as tar:
    tar.add(export_path, arcname='yolov8n.onnx')
print(f'Packaged {artifact_name} ({os.path.getsize(artifact_name) / 1e6:.2f} MB)')

# Download instructions
print('''
=== Deployment Instructions ===
1. Download artifact to your local machine.
2. Extract yolov8n.onnx into AccessGuard: data/models/yolov8n.onnx
3. AccessGuard ONNXAdapter will automatically load and accelerate live inference!
''')
